# Stage 7 – GraphML export for Gephi

**Goal:** write one GraphML file per format with all node measures attached, so the networks can be opened in
Gephi. Each file is read back and compared with the graph in memory.

**Gate:** every round-trip check must pass.

**Reads:** node and edge tables, `outputs/tables/node_metrics_<format>.csv`
**Writes:** `outputs/graphml/england_men_<format>.graphml`, `outputs/tables/validation_graphml_roundtrip.csv`

In [1]:
%run ./library/lib_00_settings.ipynb
%run ./library/lib_04_network_building.ipynb
stage_log = get_stage_logger("07_graphml")

# Measures copied onto the nodes. Whole-number measures stay whole numbers in GraphML.
MEASURES_FOR_GEPHI = ["degree", "strength", "strength_per_innings", "betweenness", "closeness_lcc",
                      "eigenvector_lcc", "clustering", "core_number", "community", "effective_size", "constraint"]
WHOLE_NUMBER_MEASURES = {"degree", "strength", "core_number", "community"}

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
def graph_with_measures(format_name: str) -> nx.Graph:
    """Format graph with the node measures added as attributes (dates as DD-MM-YYYY text)."""
    file_tag = FORMAT_FILE_TAG[format_name]
    node_table = read_table(PROCESSED_FOLDER / f"nodes_{file_tag}.csv")
    edge_table = read_table(PROCESSED_FOLDER / f"edges_{file_tag}.csv")
    for table in (node_table, edge_table):
        for date_column in ("first_date", "last_date"):
            table[date_column] = pd.to_datetime(table[date_column]).dt.strftime(DATE_TEXT_FORMAT)
    partnership_graph = build_format_graph(node_table, edge_table, format_name)
    measures_by_player = read_table(TABLES_FOLDER / f"node_metrics_{file_tag}.csv").set_index("player_id")
    for player_id in partnership_graph.nodes:
        for measure_name in MEASURES_FOR_GEPHI:
            measure_value = measures_by_player.loc[player_id, measure_name]
            if pd.isna(measure_value):
                continue   # missing values are left out, never written as the text "nan"
            partnership_graph.nodes[player_id][measure_name] = (
                int(measure_value) if measure_name in WHOLE_NUMBER_MEASURES else float(measure_value))
    return partnership_graph

In [3]:
round_trip_rows = []
for format_name in FORMATS:
    partnership_graph = graph_with_measures(format_name)
    graphml_path = GRAPHML_FOLDER / f"england_men_{FORMAT_FILE_TAG[format_name]}.graphml"
    write_graphml_file(partnership_graph, graphml_path)
    round_trip_result = graphml_round_trip_checks(partnership_graph, graphml_path)
    round_trip_rows += [{"format": format_name, "check": name, "passed": passed} for name, passed in round_trip_result.items()]
    stage_log.info("%s -> %s: %d nodes, %d edges, round trip %s", format_name, graphml_path.name,
                   partnership_graph.number_of_nodes(), partnership_graph.number_of_edges(),
                   "OK" if all(round_trip_result.values()) else f"FAILED {round_trip_result}")

round_trip_table = pd.DataFrame(round_trip_rows)
write_table(round_trip_table, TABLES_FOLDER / "validation_graphml_roundtrip.csv")
assert round_trip_table["passed"].all(), "GraphML round trip failed"

2026-11-16 20:27:32,368 INFO batnet.07_graphml: Test -> england_men_test.graphml: 51 nodes, 335 edges, round trip OK


2026-11-16 20:27:32,399 INFO batnet.07_graphml: ODI -> england_men_odi.graphml: 50 nodes, 262 edges, round trip OK


2026-11-16 20:27:32,427 INFO batnet.07_graphml: T20I -> england_men_t20i.graphml: 44 nodes, 222 edges, round trip OK
